# 10b - PyTorch Binary Two-Phase Transfer Learning

**Pipeline:** Skin Cancer Binary Cancer-Risk Classification
**Purpose:** Train EfficientNetB0 for binary cancer-risk screening using two-phase
transfer learning: (1) head-only warm-up, (2) upper-backbone fine-tuning.

**Binary mapping:**
- NV  = 0 = non_cancer
- MEL = 1 = cancer_risk
- BCC = 1 = cancer_risk

**Rules:**
- Train and val splits only. Test set is never accessed here.
- No multiclass training.
- No test evaluation, no threshold tuning.
- No image/manifest/split modifications.
- Best model selected on `val_pr_auc`.

---


## Section 0 - Imports and Environment

In [19]:
import json
import os
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE   = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE, GPU_NAME, CUDA_AVAILABLE = torch.device("cpu"), "N/A", False

print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}")
print(f"GPU         : {GPU_NAME}")
print(f"Device      : {DEVICE}")


torch       : 2.6.0+cu124
torchvision : 0.21.0+cu124
CUDA        : True
GPU         : NVIDIA GeForce RTX 4060 Laptop GPU
Device      : cuda


## Section 1 - Paths and Configuration

In [20]:
OUTPUT_ROOT        = Path(r"C:\SKIN CANCER v2\pipe output")
FINAL_DATASET_ROOT = Path(r"C:\SKIN CANCER v2\final DS")
PREPROC_DIR        = OUTPUT_ROOT / "preprocessing"
TRAINING_DIR       = OUTPUT_ROOT / "pytorch_training_binary_2phase"

BINARY_NAMES = ["non_cancer", "cancer_risk"]
BINARY_INDEX = {"NV": 0, "MEL": 1, "BCC": 1}
CLASS_NAMES  = ["NV", "MEL", "BCC"]

EXPECTED_COUNTS = {
    "train": {"total": 14332, "non_cancer": 8928, "cancer_risk": 5404},
    "val":   {"total":  3012, "non_cancer": 1886, "cancer_risk": 1126},
    "test":  {"total":  3045},
}

# Transform settings (identical to 09 and 10)
IMG_SIZE    = 224
RESIZE_TO   = 256
BATCH_SIZE  = 32
NUM_WORKERS = 0
PIN_MEMORY  = CUDA_AVAILABLE
RANDOM_SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Phase 1: head-only warm-up ────────────────────────────────────────────────
EPOCHS_P1    = 10
LR_P1        = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 5

# ── Phase 2: upper-backbone fine-tuning ───────────────────────────────────────
EPOCHS_P2 = 20
LR_P2     = 1e-4

# Reproducibility
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print("Config loaded.")
print(f"  BATCH_SIZE={BATCH_SIZE}  PATIENCE={PATIENCE}  SEED={RANDOM_SEED}")
print(f"  Phase 1: LR={LR_P1}  EPOCHS={EPOCHS_P1}")
print(f"  Phase 2: LR={LR_P2}  EPOCHS={EPOCHS_P2}")


Config loaded.
  BATCH_SIZE=32  PATIENCE=5  SEED=42
  Phase 1: LR=0.001  EPOCHS=10
  Phase 2: LR=0.0001  EPOCHS=20


## Section 2 - Create Output Folder

In [21]:
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {TRAINING_DIR}")


Ready: C:\SKIN CANCER v2\pipe output\pytorch_training_binary_2phase


## Section 3 - Load Manifests (train + val only)

In [22]:
dfs = {}
for split in ["train", "val", "test"]:
    p = PREPROC_DIR / f"{split}_manifest_preprocessed.csv"
    if not p.exists():
        raise FileNotFoundError(f"Manifest not found: {p}\nRe-run File 08 first.")
    dfs[split] = pd.read_csv(p, low_memory=False)
    dfs[split]["binary_label"] = dfs[split]["final_authoritative_label"].map(BINARY_INDEX)
    print(f"Loaded {split}: {len(dfs[split]):,} rows")

print("\n=== COUNT CHECK (train / val) ===")
for split in ["train", "val"]:
    df = dfs[split]
    bc = df["binary_label"].value_counts()
    nc = int(bc.get(0, 0))
    cr = int(bc.get(1, 0))
    ec = EXPECTED_COUNTS[split]
    row_ok = "OK" if len(df) == ec["total"] else f"DIFF exp={ec['total']:,}"
    nc_ok  = "OK" if nc == ec["non_cancer"]  else f"DIFF exp={ec['non_cancer']:,}"
    cr_ok  = "OK" if cr == ec["cancer_risk"] else f"DIFF exp={ec['cancer_risk']:,}"
    print(f"  {split}: total={len(df):,} {row_ok}  "
          f"non_cancer={nc:,} {nc_ok}  cancer_risk={cr:,} {cr_ok}")

print(f"\n  test: {len(dfs['test']):,} rows  (loaded for reference — NOT used here)")


Loaded train: 14,332 rows
Loaded val: 3,012 rows
Loaded test: 3,045 rows

=== COUNT CHECK (train / val) ===
  train: total=14,332 OK  non_cancer=8,928 OK  cancer_risk=5,404 OK
  val: total=3,012 OK  non_cancer=1,886 OK  cancer_risk=1,126 OK

  test: 3,045 rows  (loaded for reference — NOT used here)


## Section 4 - Dataset Class and Transforms

Transforms are identical to notebooks 09 and 10. No vignette cropping repeated.
No augmented images saved to disk.


In [23]:
train_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(degrees=30),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class SkinLesionDataset(Dataset):
    MULTICLASS = {"NV": 0, "MEL": 1, "BCC": 2}
    BINARY     = {"NV": 0, "MEL": 1, "BCC": 1}

    def __init__(self, df, transform=None, label_mode="binary"):
        self.df       = df.reset_index(drop=True)
        self.transform = transform
        self._lmap    = self.BINARY if label_mode == "binary" else self.MULTICLASS

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(str(row["preprocessed_full_path"])).convert("RGB")
        if self.transform:
            img = self.transform(img)
        lbl = self._lmap.get(str(row["final_authoritative_label"]), -1)
        return img, torch.tensor(lbl, dtype=torch.long)


# Smoke test
_ds = SkinLesionDataset(dfs["train"].head(4), eval_transform, "binary")
_img, _lbl = _ds[0]
print(f"Dataset smoke test: img={_img.shape}  label={_lbl.item()}  "
      f"(0=non_cancer, 1=cancer_risk)")
del _ds, _img, _lbl


Dataset smoke test: img=torch.Size([3, 224, 224])  label=0  (0=non_cancer, 1=cancer_risk)


## Section 5 - Build DataLoaders

In [24]:
def make_loader(df, transform, shuffle):
    ds = SkinLesionDataset(df, transform, "binary")
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      drop_last=False)

train_loader = make_loader(dfs["train"], train_transform, shuffle=True)
val_loader   = make_loader(dfs["val"],   eval_transform,  shuffle=False)

print(f"  train_loader: {len(train_loader.dataset):,} images  "
      f"{len(train_loader):,} batches")
print(f"  val_loader  : {len(val_loader.dataset):,} images  "
      f"{len(val_loader):,} batches")
print(f"  test data   : not loaded into a DataLoader in this notebook")


  train_loader: 14,332 images  448 batches
  val_loader  : 3,012 images  95 batches
  test data   : not loaded into a DataLoader in this notebook


## Section 6 - Model Builder and Helper Functions

**EfficientNetB0 feature blocks (indices 0–8):**
- `features[0]` — stem conv
- `features[1–4]` — lower backbone blocks (kept frozen in phase 2)
- `features[5–8]` — upper backbone blocks (unfrozen in phase 2)
- `classifier` — replaced with binary head (always trainable)


In [25]:
def build_model():
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_feat, 2),
    )
    return model.to(DEVICE)


def freeze_backbone(model):
    """Freeze all feature blocks; classifier head remains trainable."""
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_top_layers(model):
    """Unfreeze feature blocks 5-8 (upper backbone) + classifier.
    Blocks 0-4 (lower backbone / stem) stay frozen.
    """
    for i, block in enumerate(model.features):
        for param in block.parameters():
            param.requires_grad = (i >= 5)
    for param in model.classifier.parameters():
        param.requires_grad = True


def count_trainable_params(model):
    trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, non_trainable


# Verify helpers with a temp model
_m = build_model()
freeze_backbone(_m)
_t, _nt = count_trainable_params(_m)
print(f"After freeze_backbone  : trainable={_t:,}  frozen={_nt:,}")
unfreeze_top_layers(_m)
_t2, _nt2 = count_trainable_params(_m)
print(f"After unfreeze_top_layers: trainable={_t2:,}  frozen={_nt2:,}")
del _m, _t, _nt, _t2, _nt2


After freeze_backbone  : trainable=2,562  frozen=4,007,548
After unfreeze_top_layers: trainable=3,701,450  frozen=308,660


## Section 7 - Class Weights

In [26]:
def compute_class_weights(binary_label_series, n_classes=2):
    """Inverse-frequency weights normalised to n_classes (from train only)."""
    counts = binary_label_series.value_counts()
    n      = len(binary_label_series)
    w = [n / (n_classes * counts.get(i, 1)) for i in range(n_classes)]
    return torch.FloatTensor(w).to(DEVICE)

bi_weights = compute_class_weights(dfs["train"]["binary_label"])
print("Binary class weights (from train split only):")
for name, w in zip(BINARY_NAMES, bi_weights.cpu().tolist()):
    print(f"  {name}: {w:.5f}")
print(f"  non_cancer  count: {int((dfs['train']['binary_label']==0).sum()):,}")
print(f"  cancer_risk count: {int((dfs['train']['binary_label']==1).sum()):,}")


Binary class weights (from train split only):
  non_cancer: 0.80264
  cancer_risk: 1.32605
  non_cancer  count: 8,928
  cancer_risk count: 5,404


## Section 8 - Training Functions

In [27]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast("cuda", enabled=CUDA_AVAILABLE):
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def validate_binary(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out   = model(imgs)
        loss  = criterion(out, labels)
        total_loss += loss.item() * len(labels)
        probs = torch.softmax(out, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    avg_loss = total_loss / len(loader.dataset)
    acc = float((y_pred == y_true).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    try:
        roc_auc = float(roc_auc_score(y_true, y_prob))
    except Exception:
        roc_auc = 0.0
    try:
        pr_auc = float(average_precision_score(y_true, y_prob, pos_label=1))
    except Exception:
        pr_auc = 0.0
    return {
        "val_loss":             round(avg_loss, 5),
        "val_accuracy":         round(acc, 5),
        "val_precision_cancer": round(float(prec[1]), 5) if len(prec) > 1 else 0.0,
        "val_recall_cancer":    round(float(rec[1]),  5) if len(rec)  > 1 else 0.0,
        "val_f1_cancer":        round(float(f1[1]),   5) if len(f1)   > 1 else 0.0,
        "val_roc_auc":          round(roc_auc, 5),
        "val_pr_auc":           round(pr_auc,  5),
    }


print("train_one_epoch and validate_binary defined.")


train_one_epoch and validate_binary defined.


## Section 9 - Phase 1: Head-Only Training

Backbone frozen. Only the binary classifier head is trained.
`LR=1e-3`, up to 10 epochs, early stopping patience=5.


In [28]:
# ── Build model and freeze backbone ──────────────────────────────────────────
model = build_model()
freeze_backbone(model)
p1_trainable, p1_nontrain = count_trainable_params(model)
print(f"Phase 1  trainable params : {p1_trainable:,}")
print(f"Phase 1  frozen    params : {p1_nontrain:,}")

criterion = nn.CrossEntropyLoss(weight=bi_weights)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P1, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7,
)
scaler = torch.amp.GradScaler("cuda") if CUDA_AVAILABLE else None

# ── Global best-metric tracking (carries into Phase 2) ───────────────────────
best_pr_auc    = -1.0
best_roc_auc   = 0.0
best_recall    = 0.0
best_precision = 0.0
best_f1        = 0.0
best_val_loss  = float("inf")
best_epoch     = 0
best_phase     = "phase1_head"

p1_history    = []
p1_epochs_run = 0
patience_count = 0

print(f"\nPhase 1: head-only  (LR={LR_P1}  max_epochs={EPOCHS_P1}  patience={PATIENCE})")
print("=" * 80)
t_p1 = time.time()

for epoch in range(1, EPOCHS_P1 + 1):
    t0 = time.time()
    train_loss  = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    val_metrics = validate_binary(model, val_loader, criterion)
    pr_auc      = val_metrics["val_pr_auc"]
    cur_lr      = optimizer.param_groups[0]["lr"]
    scheduler.step(pr_auc)

    is_best = pr_auc > best_pr_auc
    if is_best:
        best_pr_auc    = pr_auc
        best_roc_auc   = val_metrics["val_roc_auc"]
        best_recall    = val_metrics["val_recall_cancer"]
        best_precision = val_metrics["val_precision_cancer"]
        best_f1        = val_metrics["val_f1_cancer"]
        best_val_loss  = val_metrics["val_loss"]
        best_epoch     = epoch
        best_phase     = "phase1_head"
        patience_count = 0
        torch.save({
            "epoch":          epoch,
            "global_epoch":   epoch,
            "phase":          "phase1_head",
            "model_state":    model.state_dict(),
            "optimizer":      optimizer.state_dict(),
            "best_pr_auc":    best_pr_auc,
            "val_metrics":    val_metrics,
        }, TRAINING_DIR / "best_model_binary_2phase.pt")
    else:
        patience_count += 1

    torch.save({
        "epoch":        epoch,
        "global_epoch": epoch,
        "phase":        "phase1_head",
        "model_state":  model.state_dict(),
        "optimizer":    optimizer.state_dict(),
        "val_metrics":  val_metrics,
    }, TRAINING_DIR / "last_model_binary_2phase.pt")

    row = {
        "phase":             "phase1_head",
        "epoch_in_phase":    epoch,
        "global_epoch":      epoch,
        "train_loss":        round(train_loss, 5),
        **val_metrics,
        "learning_rate":     cur_lr,
        "trainable_params":  p1_trainable,
        "non_trainable_params": p1_nontrain,
        "is_best":           is_best,
        "epoch_time_s":      round(time.time() - t0, 1),
    }
    p1_history.append(row)
    p1_epochs_run = epoch

    tag = " << BEST" if is_best else ""
    print(f"  P1 Ep {epoch:>2}/{EPOCHS_P1}  "
          f"tr={train_loss:.4f}  vl={val_metrics['val_loss']:.4f}  "
          f"pr_auc={pr_auc:.4f}  recall={val_metrics['val_recall_cancer']:.4f}  "
          f"roc={val_metrics['val_roc_auc']:.4f}  "
          f"lr={cur_lr:.2e}  {int(time.time()-t0)}s{tag}")

    if patience_count >= PATIENCE:
        print(f"  Early stopping: no pr_auc improvement for {PATIENCE} epochs")
        break

elapsed_p1 = round(time.time() - t_p1, 1)
print(f"\nPhase 1 complete in {elapsed_p1}s")
print(f"  Epochs run : {p1_epochs_run}/{EPOCHS_P1}")
print(f"  Best epoch : {best_epoch}  best_pr_auc={best_pr_auc:.5f}")


Phase 1  trainable params : 2,562
Phase 1  frozen    params : 4,007,548

Phase 1: head-only  (LR=0.001  max_epochs=10  patience=5)


KeyboardInterrupt: 

## Section 10 - Phase 2: Fine-Tuning Upper Backbone Blocks

Unfreeze EfficientNetB0 `features[5–8]` (upper blocks) + classifier.
`features[0–4]` (stem + lower blocks) remain frozen.
`LR=1e-4`, up to 20 epochs, early stopping patience=5.
Best model checkpoint continues from Phase 1 — saved whenever `val_pr_auc` improves.


In [ ]:
# ── Unfreeze upper backbone and rebuild optimizer ─────────────────────────────
unfreeze_top_layers(model)
p2_trainable, p2_nontrain = count_trainable_params(model)
print(f"Phase 2  trainable params : {p2_trainable:,}")
print(f"Phase 2  frozen    params : {p2_nontrain:,}")

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P2, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7,
)

p2_history    = []
p2_epochs_run = 0
patience_count = 0
global_offset  = p1_epochs_run

print(f"\nPhase 2: upper-backbone fine-tune  (LR={LR_P2}  max_epochs={EPOCHS_P2}  patience={PATIENCE})")
print(f"  Best pr_auc carried from Phase 1: {best_pr_auc:.5f}")
print("=" * 80)
t_p2 = time.time()

for epoch in range(1, EPOCHS_P2 + 1):
    global_epoch = global_offset + epoch
    t0 = time.time()
    train_loss  = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    val_metrics = validate_binary(model, val_loader, criterion)
    pr_auc      = val_metrics["val_pr_auc"]
    cur_lr      = optimizer.param_groups[0]["lr"]
    scheduler.step(pr_auc)

    is_best = pr_auc > best_pr_auc
    if is_best:
        best_pr_auc    = pr_auc
        best_roc_auc   = val_metrics["val_roc_auc"]
        best_recall    = val_metrics["val_recall_cancer"]
        best_precision = val_metrics["val_precision_cancer"]
        best_f1        = val_metrics["val_f1_cancer"]
        best_val_loss  = val_metrics["val_loss"]
        best_epoch     = global_epoch
        best_phase     = "phase2_finetune"
        patience_count = 0
        torch.save({
            "epoch":          epoch,
            "global_epoch":   global_epoch,
            "phase":          "phase2_finetune",
            "model_state":    model.state_dict(),
            "optimizer":      optimizer.state_dict(),
            "best_pr_auc":    best_pr_auc,
            "val_metrics":    val_metrics,
        }, TRAINING_DIR / "best_model_binary_2phase.pt")
    else:
        patience_count += 1

    torch.save({
        "epoch":        epoch,
        "global_epoch": global_epoch,
        "phase":        "phase2_finetune",
        "model_state":  model.state_dict(),
        "optimizer":    optimizer.state_dict(),
        "val_metrics":  val_metrics,
    }, TRAINING_DIR / "last_model_binary_2phase.pt")

    row = {
        "phase":             "phase2_finetune",
        "epoch_in_phase":    epoch,
        "global_epoch":      global_epoch,
        "train_loss":        round(train_loss, 5),
        **val_metrics,
        "learning_rate":     cur_lr,
        "trainable_params":  p2_trainable,
        "non_trainable_params": p2_nontrain,
        "is_best":           is_best,
        "epoch_time_s":      round(time.time() - t0, 1),
    }
    p2_history.append(row)
    p2_epochs_run = epoch

    tag = " << BEST" if is_best else ""
    print(f"  P2 Ep {epoch:>2}/{EPOCHS_P2} [G={global_epoch:>2}]  "
          f"tr={train_loss:.4f}  vl={val_metrics['val_loss']:.4f}  "
          f"pr_auc={pr_auc:.4f}  recall={val_metrics['val_recall_cancer']:.4f}  "
          f"roc={val_metrics['val_roc_auc']:.4f}  "
          f"lr={cur_lr:.2e}  {int(time.time()-t0)}s{tag}")

    if patience_count >= PATIENCE:
        print(f"  Early stopping: no pr_auc improvement for {PATIENCE} epochs")
        break

elapsed_p2 = round(time.time() - t_p2, 1)
print(f"\nPhase 2 complete in {elapsed_p2}s")
print(f"  Epochs run : {p2_epochs_run}/{EPOCHS_P2}")
print(f"  Best epoch : {best_epoch}  best_phase={best_phase}  best_pr_auc={best_pr_auc:.5f}")


Phase 2  trainable params : 3,701,450
Phase 2  frozen    params : 308,660

Phase 2: upper-backbone fine-tune  (LR=0.0001  max_epochs=20  patience=5)
  Best pr_auc carried from Phase 1: 0.81773
  P2 Ep  1/20 [G=11]  tr=0.4039  vl=0.3868  pr_auc=0.8681  recall=0.7726  roc=0.9010  lr=1.00e-04  161s << BEST
  P2 Ep  2/20 [G=12]  tr=0.3382  vl=0.3721  pr_auc=0.8822  recall=0.7806  roc=0.9131  lr=1.00e-04  160s << BEST
  P2 Ep  3/20 [G=13]  tr=0.3081  vl=0.3485  pr_auc=0.8930  recall=0.7966  roc=0.9197  lr=1.00e-04  160s << BEST
  P2 Ep  4/20 [G=14]  tr=0.2822  vl=0.4030  pr_auc=0.8851  recall=0.8428  roc=0.9196  lr=1.00e-04  160s
  P2 Ep  5/20 [G=15]  tr=0.2585  vl=0.3645  pr_auc=0.8929  recall=0.8099  roc=0.9219  lr=1.00e-04  160s
  P2 Ep  6/20 [G=16]  tr=0.2344  vl=0.3555  pr_auc=0.8909  recall=0.7993  roc=0.9256  lr=1.00e-04  160s
  P2 Ep  7/20 [G=17]  tr=0.2117  vl=0.3580  pr_auc=0.9023  recall=0.8153  roc=0.9295  lr=5.00e-05  161s << BEST
  P2 Ep  8/20 [G=18]  tr=0.1996  vl=0.3669  pr_

## Section 11 - Combine History and Save Artifacts

In [ ]:
# Combined training history
history = pd.DataFrame(p1_history + p2_history)
hist_cols = [
    "phase", "epoch_in_phase", "global_epoch",
    "train_loss", "val_loss", "val_accuracy",
    "val_precision_cancer", "val_recall_cancer", "val_f1_cancer",
    "val_roc_auc", "val_pr_auc",
    "learning_rate", "trainable_params", "non_trainable_params",
]
history = history[[c for c in hist_cols if c in history.columns]]
hist_path = TRAINING_DIR / "training_history_binary_2phase.csv"
history.to_csv(hist_path, index=False)
print(f"Saved {hist_path.name}  ({len(history)} total epochs)")

# Training config JSON
config = {
    "model":              "efficientnet_b0",
    "label_mode":         "binary",
    "n_classes":          2,
    "binary_names":       BINARY_NAMES,
    "binary_mapping":     {"NV": 0, "MEL": 1, "BCC": 1},
    "img_size":           IMG_SIZE,
    "batch_size":         BATCH_SIZE,
    "weight_decay":       WEIGHT_DECAY,
    "early_stop_patience": PATIENCE,
    "optimizer":          "AdamW",
    "scheduler":          "ReduceLROnPlateau(mode=max, factor=0.5, patience=2)",
    "random_seed":        RANDOM_SEED,
    "phase1": {
        "description":       "head-only warm-up (backbone frozen)",
        "frozen_blocks":     "features[0-8] (all)",
        "trainable":         "classifier only",
        "lr":                LR_P1,
        "max_epochs":        EPOCHS_P1,
        "epochs_run":        p1_epochs_run,
        "trainable_params":  p1_trainable,
        "non_trainable_params": p1_nontrain,
    },
    "phase2": {
        "description":       "upper-backbone fine-tuning",
        "frozen_blocks":     "features[0-4]",
        "trainable":         "features[5-8] + classifier",
        "lr":                LR_P2,
        "max_epochs":        EPOCHS_P2,
        "epochs_run":        p2_epochs_run,
        "trainable_params":  p2_trainable,
        "non_trainable_params": p2_nontrain,
    },
    "class_weight_non_cancer":  round(bi_weights[0].item(), 5),
    "class_weight_cancer_risk": round(bi_weights[1].item(), 5),
    "best_epoch":       best_epoch,
    "best_phase":       best_phase,
    "best_val_pr_auc":  round(best_pr_auc,    5),
    "best_val_roc_auc": round(best_roc_auc,   5),
    "best_val_recall_cancer":    round(best_recall,    5),
    "best_val_precision_cancer": round(best_precision, 5),
    "best_val_f1_cancer":        round(best_f1,        5),
    "best_val_loss":    round(best_val_loss,  5),
    "device":           str(DEVICE),
    "gpu_name":         GPU_NAME,
    "test_set_used":    False,
    "images_modified":  False,
    "manifests_modified": False,
}
cfg_path = TRAINING_DIR / "training_config_binary_2phase.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print(f"Saved {cfg_path.name}")


Saved training_history_binary_2phase.csv  (22 total epochs)
Saved training_config_binary_2phase.json


## Section 12 - Training Curves

Four panels: Loss · PR-AUC · Cancer Recall · Cancer F1 / ROC-AUC
Dashed vertical line marks the phase boundary. Best epoch marked.


In [ ]:
phase_boundary = p1_epochs_run + 0.5  # x-position between phase 1 and phase 2
gep = history["global_epoch"]
p1_mask = history["phase"] == "phase1_head"
p2_mask = history["phase"] == "phase2_finetune"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

def vlines(ax):
    ax.axvline(phase_boundary, ls=":", color="steelblue", alpha=0.7,
               label="phase boundary")
    ax.axvline(best_epoch, ls="--", color="gray", alpha=0.6,
               label=f"best ep {best_epoch}")

# Loss
ax = axes[0, 0]
ax.plot(gep, history["train_loss"], marker="o", ms=3, label="train_loss")
ax.plot(gep, history["val_loss"],   marker="s", ms=3, label="val_loss")
vlines(ax)
ax.set_title("Loss"); ax.legend(fontsize=8); ax.set_xlabel("Global Epoch")

# PR-AUC
ax = axes[0, 1]
ax.plot(gep, history["val_pr_auc"],  marker="o", ms=3, color="tab:green",  label="val_pr_auc")
ax.plot(gep, history["val_roc_auc"], marker="s", ms=3, color="tab:olive",  label="val_roc_auc")
vlines(ax)
ax.set_title("Val AUC Metrics"); ax.legend(fontsize=8); ax.set_xlabel("Global Epoch")

# Cancer Recall
ax = axes[1, 0]
ax.plot(gep, history["val_recall_cancer"],    marker="o", ms=3, color="tab:red",    label="val_recall_cancer")
ax.plot(gep, history["val_precision_cancer"], marker="s", ms=3, color="tab:orange", label="val_precision_cancer")
vlines(ax)
ax.set_title("Cancer-Risk Recall / Precision")
ax.legend(fontsize=8); ax.set_xlabel("Global Epoch")

# F1
ax = axes[1, 1]
ax.plot(gep, history["val_f1_cancer"],  marker="o", ms=3, color="tab:purple", label="val_f1_cancer")
ax.plot(gep, history["val_accuracy"],   marker="s", ms=3, color="tab:blue",   label="val_accuracy")
vlines(ax)
ax.set_title("Cancer F1 / Val Accuracy")
ax.legend(fontsize=8); ax.set_xlabel("Global Epoch")

# Phase labels
for ax in axes.flat:
    ymin, ymax = ax.get_ylim()
    mid_y = (ymin + ymax) * 0.98
    if p1_epochs_run > 0:
        ax.text(p1_epochs_run / 2, mid_y, "Phase 1\nHead", ha="center",
                fontsize=7, color="steelblue", va="top")
    if p2_epochs_run > 0:
        p2_mid = p1_epochs_run + p2_epochs_run / 2
        ax.text(p2_mid, mid_y, "Phase 2\nFine-tune", ha="center",
                fontsize=7, color="darkorange", va="top")

plt.suptitle(
    f"Binary Two-Phase Training — EfficientNetB0  "
    f"(best ep {best_epoch}, {best_phase}, pr_auc={best_pr_auc:.4f})",
    fontsize=12,
)
plt.tight_layout()
curve_path = TRAINING_DIR / "training_curves_binary_2phase.png"
plt.savefig(str(curve_path), dpi=100, bbox_inches="tight")
plt.close()
print(f"Saved {curve_path.name}")


C:\Users\Ahmed Hatem\AppData\Local\Temp\ipykernel_32884\636503436.py:61: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Saved training_curves_binary_2phase.png


## Section 13 - Training Summary CSV

In [ ]:
summary_df = pd.DataFrame([{
    "model":                    "binary_2phase",
    "best_epoch":               best_epoch,
    "best_phase":               best_phase,
    "best_val_pr_auc":          round(best_pr_auc,    5),
    "best_val_roc_auc":         round(best_roc_auc,   5),
    "best_val_recall_cancer":   round(best_recall,    5),
    "best_val_precision_cancer": round(best_precision,5),
    "best_val_f1_cancer":       round(best_f1,        5),
    "best_val_loss":            round(best_val_loss,  5),
    "phase1_epochs_run":        p1_epochs_run,
    "phase2_epochs_run":        p2_epochs_run,
    "total_epochs_run":         p1_epochs_run + p2_epochs_run,
    "device":                   str(DEVICE),
    "gpu_name":                 GPU_NAME,
    "test_set_used":            False,
}])
summary_path = TRAINING_DIR / "binary_2phase_training_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved {summary_path.name}")
from IPython.display import display
display(summary_df)


Saved binary_2phase_training_summary.csv


,model,best_epoch,best_phase,best_val_pr_auc,best_val_roc_auc,best_val_recall_cancer,best_val_precision_cancer,best_val_f1_cancer,best_val_loss,phase1_epochs_run,phase2_epochs_run,total_epochs_run,device,gpu_name,test_set_used
0,binary_2phase,17,phase2_finetune,0.90232,0.92955,0.81528,0.80245,0.80881,0.35803,10,12,22,cuda,NVIDIA GeForce RTX 4060 Laptop GPU,False


## Section 14 - Output File Verification

In [ ]:
required_files = [
    TRAINING_DIR / "best_model_binary_2phase.pt",
    TRAINING_DIR / "last_model_binary_2phase.pt",
    TRAINING_DIR / "training_history_binary_2phase.csv",
    TRAINING_DIR / "training_curves_binary_2phase.png",
    TRAINING_DIR / "training_config_binary_2phase.json",
    TRAINING_DIR / "binary_2phase_training_summary.csv",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<45} {size:>12,} bytes")
    if not exists:
        all_ok = False
print("\nAll required files present." if all_ok else "\nWARNING: missing files above.")


Output file verification:
  [OK] best_model_binary_2phase.pt                     46,054,967 bytes
  [OK] last_model_binary_2phase.pt                     46,054,903 bytes
  [OK] training_history_binary_2phase.csv                   2,479 bytes
  [OK] training_curves_binary_2phase.png                  174,318 bytes
  [OK] training_config_binary_2phase.json                   1,522 bytes
  [OK] binary_2phase_training_summary.csv                     364 bytes

All required files present.


## Section 15 - Final Summary (Copy-Paste Ready)

In [ ]:
from IPython.display import display

print("=" * 72)
print("  10_pytorch_binary_two_phase_training -- FINAL SUMMARY")
print("=" * 72)

print(f"\n 1. torch version      : {torch.__version__}")
print(f" 2. CUDA available     : {CUDA_AVAILABLE}")
print(f" 3. GPU name           : {GPU_NAME}")
print(f" 4. Device used        : {DEVICE}")

print(f"\n 5. Train rows         : {len(dfs['train']):,}")
print(f" 6. Val rows           : {len(dfs['val']):,}")

print(f"\n 7. Train binary counts:")
for name, idx in [("non_cancer", 0), ("cancer_risk", 1)]:
    n = int((dfs["train"]["binary_label"] == idx).sum())
    print(f"      {name}: {n:,}")
print(f"    8. Val binary counts:")
for name, idx in [("non_cancer", 0), ("cancer_risk", 1)]:
    n = int((dfs["val"]["binary_label"] == idx).sum())
    print(f"      {name}: {n:,}")

print(f"\n 9. Binary class weights:")
for name, w in zip(BINARY_NAMES, bi_weights.cpu().tolist()):
    print(f"      {name}: {w:.5f}")

print(f"\n10. Phase 1 planned epochs : {EPOCHS_P1}")
print(f"    Phase 1 actual  epochs : {p1_epochs_run}")
print(f"11. Phase 2 planned epochs : {EPOCHS_P2}")
print(f"    Phase 2 actual  epochs : {p2_epochs_run}")

print(f"\n12. Phase 1 trainable params : {p1_trainable:,}")
print(f"    Phase 1 frozen    params : {p1_nontrain:,}")
print(f"13. Phase 2 trainable params : {p2_trainable:,}")
print(f"    Phase 2 frozen    params : {p2_nontrain:,}")

print(f"\n14. Best global epoch   : {best_epoch}")
print(f"15. Best phase          : {best_phase}")
print(f"16. Best val_pr_auc     : {best_pr_auc:.5f}")
print(f"17. Best val_roc_auc    : {best_roc_auc:.5f}")
print(f"18. Best val_recall_cancer    : {best_recall:.5f}")
print(f"19. Best val_precision_cancer : {best_precision:.5f}")
print(f"20. Best val_f1_cancer        : {best_f1:.5f}")
print(f"21. Best val_loss             : {best_val_loss:.5f}")

print(f"\n22. Output file verification:")
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"      [{status}] {p.name:<45} {size:>10,} bytes")

print(f"\n23. Test set used for training          : False")
print(f"    Test set used for model selection   : False")
print(f"    Test set used for threshold tuning  : False")
print(f"24. Images modified                     : False")
print(f"25. Manifests modified                  : False")
print(f"    Splits modified                     : False")

print(f"\nPhase 1 epoch history:")
p1_df = pd.DataFrame(p1_history)
display_cols = ["epoch_in_phase","train_loss","val_loss","val_pr_auc",
                "val_recall_cancer","val_roc_auc","learning_rate","is_best"]
display(p1_df[[c for c in display_cols if c in p1_df.columns]])

print(f"\nPhase 2 epoch history:")
p2_df = pd.DataFrame(p2_history)
display(p2_df[[c for c in display_cols if c in p2_df.columns]])

print("=" * 72)


  10_pytorch_binary_two_phase_training -- FINAL SUMMARY

 1. torch version      : 2.6.0+cu124
 2. CUDA available     : True
 3. GPU name           : NVIDIA GeForce RTX 4060 Laptop GPU
 4. Device used        : cuda

 5. Train rows         : 14,332
 6. Val rows           : 3,012

 7. Train binary counts:
      non_cancer: 8,928
      cancer_risk: 5,404
    8. Val binary counts:
      non_cancer: 1,886
      cancer_risk: 1,126

 9. Binary class weights:
      non_cancer: 0.80264
      cancer_risk: 1.32605

10. Phase 1 planned epochs : 10
    Phase 1 actual  epochs : 10
11. Phase 2 planned epochs : 20
    Phase 2 actual  epochs : 12

12. Phase 1 trainable params : 2,562
    Phase 1 frozen    params : 4,007,548
13. Phase 2 trainable params : 3,701,450
    Phase 2 frozen    params : 308,660

14. Best global epoch   : 17
15. Best phase          : phase2_finetune
16. Best val_pr_auc     : 0.90232
17. Best val_roc_auc    : 0.92955
18. Best val_recall_cancer    : 0.81528
19. Best val_precision_c

,epoch_in_phase,train_loss,val_loss,val_pr_auc,val_recall_cancer,val_roc_auc,learning_rate,is_best
0,1,0.48730,0.46871,0.80520,0.80373,0.86510,0.001,True
1,2,0.45124,0.45890,0.81315,0.79485,0.86947,0.001,True
2,3,0.45410,0.46285,0.81388,0.80639,0.87064,0.001,True
3,4,0.44997,0.44813,0.81501,0.78952,0.87216,0.001,True
4,5,0.44759,0.48402,0.80882,0.83215,0.87118,0.001,False
5,6,0.45398,0.47539,0.81247,0.82327,0.87017,0.001,False
6,7,0.45073,0.46434,0.81583,0.79396,0.86744,0.001,True
7,8,0.44914,0.46282,0.81773,0.81261,0.87127,0.001,True
8,9,0.45000,0.49137,0.81348,0.84369,0.87183,0.001,False
9,10,0.44599,0.49468,0.80817,0.84813,0.86721,0.001,False



Phase 2 epoch history:


,epoch_in_phase,train_loss,val_loss,val_pr_auc,val_recall_cancer,val_roc_auc,learning_rate,is_best
0,1,0.40390,0.38680,0.86808,0.77265,0.90095,0.000100,True
1,2,0.33823,0.37206,0.88222,0.78064,0.91305,0.000100,True
2,3,0.30813,0.34849,0.89304,0.79663,0.91973,0.000100,True
3,4,0.28224,0.40298,0.88506,0.84281,0.91961,0.000100,False
4,5,0.25855,0.36451,0.89287,0.80995,0.92192,0.000100,False
5,6,0.23441,0.35546,0.89092,0.79929,0.92565,0.000100,False
6,7,0.21168,0.35803,0.90232,0.81528,0.92955,0.000050,True
7,8,0.19959,0.36685,0.89858,0.80906,0.92754,0.000050,False
8,9,0.19092,0.37819,0.89471,0.76732,0.92439,0.000050,False
9,10,0.17902,0.38965,0.89341,0.80906,0.92601,0.000050,False


## Section 16 - Completion Summary

**10_pytorch_binary_two_phase_training is complete.**

**What was accomplished:**
- EfficientNetB0 loaded with ImageNet pretrained weights.
- Binary head (2 outputs) replaces default classifier.
- Phase 1 (head-only): backbone frozen, only classifier trained with `LR=1e-3`.
- Phase 2 (fine-tuning): `features[5–8]` + classifier unfrozen, trained with `LR=1e-4`.
- Class-weighted CrossEntropyLoss (computed from train split only).
- Best model checkpoint saved on `val_pr_auc` across both phases.
- Mixed precision training via `torch.amp`.
- Early stopping (patience=5) in each phase.

**What was deliberately deferred:**
- Test set evaluation
- Decision threshold tuning
- Calibration
- Confusion matrices
- Grad-CAM / explainability
- Multiclass training

**Next step:** Binary evaluation notebook — load `best_model_binary_2phase.pt`,
evaluate on the test set, tune thresholds, report final metrics.
